# Espectroscopía de dos tonos con flujo (bobina) y drive por la antena

Para cada corriente de la bobina:
1. la fuente rampa hasta la corriente y espera;
2. con el **drive apagado**, el VNA barre alrededor del resonador y el **marcador busca el mínimo de |S21|**
   (`FUNC MIN` con tracking sobre la traza S21/MLOG); con `CALC:MEAS:MARK:SET CWFReq` el VNA copia la frecuencia
   del marcador a su **frecuencia CW**;
3. el VNA pasa a **CW** en esa frecuencia y se mide una referencia sin drive;
4. se enciende el SG y se barre el drive (potencia × frecuencia) leyendo el VNA en CW.

El mínimo se busca **solo al cambiar la corriente de la bobina**, nunca al cambiar la frecuencia o la potencia del drive.
Cada corriente es un run propio de QCoDeS (con su `run_id` y su metadata) en la base de la carpeta `two_tone_flux2`.
Funciones en `dos_tonos.py`.

In [ ]:
# --- Rutas del repositorio (celda agregada al subir a GitHub) ---
# El código está en ../codigo y las bases de datos en ../datos. Esta celda agrega ../codigo al path
# y se para en ../datos, así las rutas relativas de las bases (p. ej. 'fluxonium_2/punch_out.db')
# funcionan igual que en la PC del laboratorio. Se puede correr más de una vez.
import os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'codigo').is_dir() and (p / 'datos').is_dir())
if str(REPO / 'codigo') not in sys.path:
    sys.path.insert(0, str(REPO / 'codigo'))
os.chdir(REPO / 'datos')
print('repo:', REPO, '| cwd:', os.getcwd())

In [1]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import qcodes as qc
from qcodes.dataset import initialise_or_create_database_at, load_or_create_experiment, load_by_id
from qcodes.instrument_drivers.Keysight import KeysightN5245A
from qcodes.instrument_drivers.Keysight.Keysight_N5173B import KeysightN5173B

from barrido_flujo import conectar_fuente
from dos_tonos import medir_dos_tonos, tiempo_estimado, cargar_dos_tonos, buscar_dos_tonos

%matplotlib tk

### Base de datos

In [2]:
carpeta_db = os.path.abspath('two_tone_flux2')
os.makedirs(carpeta_db, exist_ok=True)

path2database = os.path.join(carpeta_db, 'two_tone_flux2.db')
initialise_or_create_database_at(path2database)
exp = load_or_create_experiment(experiment_name='dos_tonos', sample_name='fluxonium')
print('base:', qc.config.core.db_location)
print(exp)

base: c:\Users\Leandro\Documents\RF\rf-2026\qubits\two_tone_flux2\two_tone_flux2.db
dos_tonos#fluxonium#1@c:\Users\Leandro\Documents\RF\rf-2026\qubits\two_tone_flux2\two_tone_flux2.db
---------------------------------------------------------------------------------------------------
1-dos_tonos_Q4_I-139.40uA-1-p_drive,f_drive,f_busqueda,s21_re,s21_im,s21_mag,s21_phase,s21_err,busqueda_re,busqueda_im,busqueda_mag-13468
2-dos_tonos_Q4_I-139.40uA-2-p_drive,f_drive,f_busqueda,s21_re,s21_im,s21_mag,s21_phase,s21_err,busqueda_re,busqueda_im,busqueda_mag-13468
3-dos_tonos_Q4_I-139.40uA-3-p_drive,f_drive,f_busqueda,s21_re,s21_im,s21_mag,s21_phase,s21_err,busqueda_re,busqueda_im,busqueda_mag-648
4-dos_tonos_Q4_I-139.40uA-4-p_drive,f_drive,f_busqueda,s21_re,s21_im,s21_mag,s21_phase,s21_err,busqueda_re,busqueda_im,busqueda_mag-10098
5-dos_tonos_Q4_I-139.40uA-5-p_drive,f_drive,f_busqueda,s21_re,s21_im,s21_mag,s21_phase,s21_err,busqueda_re,busqueda_im,busqueda_mag-7038
6-dos_tonos_Q4_I-139.40uA-6-p

### Conexión: PNA, generador (drive por la antena) y fuente de la bobina
Si otro notebook (`punch_out.ipynb`, `readout.ipynb`) tiene abiertos el PNA, el SG o la Keithley, cerralos ahí primero.
`I_max` es un límite de seguridad: la fuente no acepta corrientes mayores.

In [3]:
addr_pna = 'TCPIP::192.168.1.6::inst0::INSTR'
pna = KeysightN5245A('pna', address=addr_pna)

sg = KeysightN5173B('sg', address='GPIB1::19::INSTR')
sg.rf_output(False)                                             # apagado hasta que empiece la medición

station = qc.Station(pna, sg)

fuente = conectar_fuente('GPIB0::15::INSTR', I_max=300e-6)     # A
print(pna.ask('*IDN?').strip())
print(sg.ask('*IDN?').strip(), f'| f = {sg.frequency()/1e9:.6f} GHz, P = {sg.power():.2f} dBm, salida {sg.rf_output()}')
print(f'corriente actual de la bobina: {fuente.actual() * 1e6:.3f} uA')

Connected to: Keysight Technologies N5242B (serial:MY63056409, firmware:A.15.71.08) in 0.14s
Connected to: Agilent Technologies N5173B (serial:MY62220514, firmware:B.01.96) in 0.09s


[pna_port3(KeysightPNAPort)] Snapshot: Could not update parameter: source_power
[pna_port4(KeysightPNAPort)] Snapshot: Could not update parameter: source_power
[pna_port3(KeysightPNAPort)] Snapshot: Could not update parameter: source_power
[pna_port4(KeysightPNAPort)] Snapshot: Could not update parameter: source_power


Keysight Technologies,N5242B,MY63056409,A.15.71.08
Agilent Technologies, N5173B, MY62220514, B.01.96 | f = 4.040000 GHz, P = 3.00 dBm, salida False
corriente actual de la bobina: -0.005 uA


### Parámetros
- **Bobina**: corrientes a medir (un run por corriente). Q4 tiene histéresis: el estado n = 0 se prepara rampeando
  hasta −200 µA y subiendo (`preparar_desde`).
- **Búsqueda del mínimo**: barrido LIN del VNA con el drive apagado; el marcador deja la f CW en el mínimo.
  El VNA necesita una traza **S21 en formato MLOG** (sobre esa busca el marcador 1).
- **Lectura CW**: `puntos_cw` puntos promediados en cada frecuencia del drive (`puntos_cw / ifbw_cw` s por punto).
- **Drive**: frecuencias y potencias del SG (las potencias son el lazo externo).

In [7]:
# ---- bobina (flujo) ----
corrientes     = np.arange(-0.005e-6, -2.0e-6, -0.005e-6)    # A; varias -> un run por corriente, p.ej. np.arange(-160e-6, -114e-6, 2e-6)
preparar_desde = -0.005e-6                  # A, rampa previa (Q4: n = 0 desde -200 uA y subiendo); None = no
paso_rampa     = 0.005e-6                     # A, paso de la rampa entre corrientes
espera_I       = 0.2                      # s, espera después de llegar a cada corriente
volver_a       = None                     # A, corriente al terminar (None = queda en la última)

# ---- VNA: búsqueda del mínimo (solo al cambiar la corriente de la bobina) ----
centro_busqueda = 5.99e9    # Hz (res_q4)
span_busqueda   = 30e6        # Hz (~4 κ)
puntos_busqueda = 201
ifbw_busqueda   = 10         # Hz -> ~2 s por búsqueda
seguir_minimo   = True        # la búsqueda de la próxima corriente se centra en el mínimo anterior

# ---- VNA: lectura CW ----
p_vna             = -20       # dBm, la misma en la búsqueda y en CW (con -112 dB: ~2-3 fotones en res_q4)
puntos_cw         = 1        # puntos CW que se promedian en cada f del drive
ifbw_cw           = 10       # Hz -> puntos_cw / ifbw_cw = 0.1 s por f del drive
desplazamiento_cw = 0.0       # Hz, f CW = mínimo + desplazamiento (0 = en el mínimo)
buscar_cada       = 0

# ---- drive (SG por la antena) ----
f_drive      = np.arange(1e6, 4e9, 5e6)   # Hz (el máximo depende de la opción del N5173B)
p_drive      = [5]                             # dBm a la salida del SG (lista -> lazo externo)
espera_drive = 0.01           # s, después de cambiar la f del SG
excluir_cw   = 20e6           # Hz, no mide el drive a menos de esto de la f CW (None = mide todo)
n_ref        = 5              # barridos CW sin drive para la referencia de cada corriente

# ---- metadata ----
resonador = 'res_q4'
qubit     = 'Q4'
att_vna   = -62 - 50          # dB, atenuación TOTAL de entrada: 2 externos (-62) + línea del crióstato (-50)
att_drive = '20'       # dB, atenuación del SG hasta la antena
notas     = 'dos tonos Q4, drive por la antena'

t_tot, t_pt = tiempo_estimado(len(corrientes), len(p_drive), len(f_drive), puntos_cw, ifbw_cw, espera_drive,
                              puntos_busqueda, ifbw_busqueda, espera_I, n_ref, 0.05, buscar_cada)
print(f'{len(corrientes)} corriente(s) x {len(p_drive)} potencia(s) x {len(f_drive)} frecuencias | '
      f'~{t_pt * 1e3:.0f} ms por punto -> ~{t_tot / 60:.1f} min')

399 corriente(s) x 1 potencia(s) x 800 frecuencias | ~160 ms por punto -> ~857.9 min


## Medición en vivo
Se abre una ventana (matplotlib tk) que se actualiza mientras se mide:
- arriba: la traza de búsqueda con el mínimo y la f CW; la f CW en cada corriente;
- medio: Δ|S21| y Δfase respecto de la referencia sin drive, vs la frecuencia del drive;
- abajo: los mapas (eje y = corriente de la bobina si hay más de una, si no la potencia del drive).

Para cortar: interrumpí el kernel (■). Lo medido queda guardado, el SG se apaga y el VNA vuelve a barrer el resonador.

In [ ]:
res = medir_dos_tonos(
    pna, sg, fuente, corrientes, f_drive, p_drive,
    centro_busqueda=centro_busqueda, span_busqueda=span_busqueda,
    puntos_busqueda=puntos_busqueda, ifbw_busqueda=ifbw_busqueda, seguir_minimo=seguir_minimo,
    p_vna=p_vna, puntos_cw=puntos_cw, ifbw_cw=ifbw_cw, desplazamiento_cw=desplazamiento_cw,
    espera_drive=espera_drive, excluir_cw=excluir_cw, n_ref=n_ref,
    paso_rampa=paso_rampa, espera_I=espera_I, preparar_desde=preparar_desde, volver_a=volver_a,
    vivo=True, nombre=f'dos_tonos_{qubit}',
    resonador=resonador, qubit=qubit, att_vna=att_vna, att_drive=att_drive, notas=notas)
runs_dt = [r['run_id'] for r in res['runs']]
print('runs:', runs_dt)

dos tonos 20261112_092025: 399 corriente(s) x 1 potencia(s) x 800 frecuencias del drive, 1 búsqueda(s) del mínimo | ~160 ms por punto -> ~857.9 min (sin la rampa de la bobina)
preparando: rampa a -0.01 uA
I =   -0.005 uA | mínimo 5.990150 GHz (-71.57 dB) | f CW 5.990150 GHz | ref -71.24 dB
Starting experimental run with id: 620. 
  run 620 listo
I =   -0.010 uA | f CW fija 5.990150 GHz | ref -71.19 dB, +79.35°
Starting experimental run with id: 621. 
  run 621 listo
I =   -0.015 uA | f CW fija 5.990150 GHz | ref -71.11 dB, +77.92°
Starting experimental run with id: 622. 
  run 622 listo
I =   -0.020 uA | f CW fija 5.990150 GHz | ref -71.09 dB, +81.73°
Starting experimental run with id: 623. 
  run 623 listo
I =   -0.025 uA | f CW fija 5.990150 GHz | ref -71.03 dB, +79.86°
Starting experimental run with id: 624. 
  run 624 listo
I =   -0.030 uA | f CW fija 5.990150 GHz | ref -71.41 dB, +81.09°
Starting experimental run with id: 625. 
  run 625 listo
I =   -0.035 uA | f CW fija 5.990150 

## Un tono: barrido de corriente leyendo en CW (sin drive)
El SG queda apagado y solo se barre la corriente de la bobina; en cada corriente el VNA lee S21 en CW
(`puntos_cw_1t` puntos promediados). Todo el barrido es **un solo run** con setpoint `I_bobina`.

Cómo se elige la f CW:
- `buscar_cada_1t = 0`: busca el mínimo solo en la primera corriente y deja la f CW fija (el corrimiento del
  resonador con el flujo se ve como cambio de fase/amplitud en CW);
- `buscar_cada_1t = n`: vuelve a buscar el mínimo cada n corrientes (1 = en todas, la f CW sigue al resonador);
- `f_cw_1t = ...` [Hz]: f CW fija en ese valor, sin buscar nunca.

La ventana en vivo muestra la búsqueda, el mínimo vs I, y |S21| y fase en CW vs I. Para cortar: ■ (lo medido queda guardado).

In [17]:
import importlib, dos_tonos
importlib.reload(dos_tonos)                  # por si el kernel ya tenía cargada la versión vieja
from dos_tonos import medir_un_tono, cargar_un_tono, tiempo_estimado_un_tono

# ---- bobina (flujo) ----
corrientes_1t     = np.arange(-0.005e-6, -2.0e-6, -0.005e-6)   # A (10 nA ~ 0.015 Φ0 si el período es ~0.69 uA)
preparar_desde_1t = -1.995e-6                  # A, rampa previa (siempre se llega desde el mismo lado); None = no
paso_rampa_1t     = 0.005e-6                    # A
espera_I_1t       = 0.2                      # s, espera después de llegar a cada corriente
volver_a_1t       = None                     # A, corriente al terminar (None = queda en la última)

# ---- VNA ----
centro_busqueda_1t = 6.07e9                 # Hz (res_q1)
span_busqueda_1t   = 30e6                    # Hz
puntos_busqueda_1t = 201
ifbw_busqueda_1t   = 10                      # Hz
p_vna_1t           = -20                     # dBm (búsqueda y CW)
puntos_cw_1t       = 3                     # puntos CW promediados en cada corriente
ifbw_cw_1t         = 1                     # Hz -> puntos_cw_1t / ifbw_cw_1t = 1 s por corriente
buscar_cada_1t     = 0                       # 0 = f CW fija (mínimo de la primera corriente); n = cada n corrientes
f_cw_1t            = None                    # Hz; si se da, f CW fija ahí y no se busca nunca

# ---- metadata ----
resonador_1t = 'res_q2'
qubit_1t     = 'Q2'
notas_1t     = 'un tono: barrido fino de corriente, S21 en CW, sin drive'

n_b = 0 if f_cw_1t is not None else (1 if buscar_cada_1t == 0 else int(np.ceil(len(corrientes_1t) / buscar_cada_1t)))
t1 = tiempo_estimado_un_tono(len(corrientes_1t), puntos_cw_1t, ifbw_cw_1t, espera_I_1t, n_b,
                             puntos_busqueda_1t, ifbw_busqueda_1t)
print(f'{len(corrientes_1t)} corrientes de {corrientes_1t[0]*1e6:.3f} a {corrientes_1t[-1]*1e6:.3f} uA, '
      f'{n_b} búsqueda(s) -> ~{t1 / 60:.1f} min')

399 corrientes de -0.005 a -1.995 uA, 1 búsqueda(s) -> ~22.3 min


In [18]:
res_1t = medir_un_tono(
    pna, fuente, corrientes_1t,
    centro_busqueda=centro_busqueda_1t, span_busqueda=span_busqueda_1t,
    puntos_busqueda=puntos_busqueda_1t, ifbw_busqueda=ifbw_busqueda_1t, p_vna=p_vna_1t,
    puntos_cw=puntos_cw_1t, ifbw_cw=ifbw_cw_1t, f_cw=f_cw_1t, buscar_cada=buscar_cada_1t,
    paso_rampa=paso_rampa_1t, espera_I=espera_I_1t, preparar_desde=preparar_desde_1t, volver_a=volver_a_1t,
    sg=sg, vivo=True, nombre=f'un_tono_{qubit_1t}',
    resonador=resonador_1t, qubit=qubit_1t, att_vna=globals().get('att_vna', ''), notas=notas_1t)
print('run:', res_1t['run_id'])

un tono 20261111_165717: 399 corrientes, f CW fija (mínimo buscado en la primera corriente), 1 búsqueda(s) -> ~22.3 min (sin la rampa de la bobina)
preparando: rampa a -1.99 uA
Starting experimental run with id: 238. 
I =   -0.0050 uA | mínimo 6.069850 GHz (-72.99 dB) | f CW 6.069850 GHz
  19/399  I =   -0.0950 uA  |S21|  -72.57 dB  fase  -56.88°  | 2.1 min, faltan ~42.2 min
  38/399  I =   -0.1900 uA  |S21|  -72.55 dB  fase  -57.27°  | 3.2 min, faltan ~29.9 min
  57/399  I =   -0.2850 uA  |S21|  -72.48 dB  fase  -57.12°  | 4.2 min, faltan ~25.1 min
  76/399  I =   -0.3800 uA  |S21|  -72.50 dB  fase  -57.57°  | 5.2 min, faltan ~22.3 min
  95/399  I =   -0.4750 uA  |S21|  -72.52 dB  fase  -58.11°  | 6.3 min, faltan ~20.1 min
  114/399  I =   -0.5700 uA  |S21|  -72.66 dB  fase  -57.35°  | 7.3 min, faltan ~18.3 min
  133/399  I =   -0.6650 uA  |S21|  -72.63 dB  fase  -57.69°  | 8.3 min, faltan ~16.7 min
  152/399  I =   -0.7600 uA  |S21|  -72.64 dB  fase  -58.48°  | 9.4 min, faltan ~15.2 

In [7]:
# cargar y graficar un run de un tono (cualquier run_id)
run_1t = res_1t['run_id']
d1 = cargar_un_tono(run_1t)
I_uA = d1['I'] * 1e6
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax[0].plot(I_uA, 20 * np.log10(np.abs(d1['z'])), '.-', ms=3, lw=.6)
ax[0].set_ylabel('|S21| CW [dB]')
ax[1].plot(I_uA, np.degrees(np.unwrap(np.angle(d1['z']))), '.-', ms=3, lw=.6, color='C3')
ax[1].set_ylabel('fase CW [°]')
ok = np.isfinite(d1['f_min'])
if ok.any():
    ax[2].plot(I_uA[ok], (d1['f_min'][ok] - d1['f_min'][ok][0]) / 1e3, 'o-', ms=3, lw=.6)
ax[2].plot(I_uA, (d1['f_cw'] - d1['f_cw'][0]) / 1e3, 'k:', lw=.8, label='f CW')
ax[2].set_ylabel('f − primera [kHz]'); ax[2].legend(fontsize=8)
ax[-1].set_xlabel('I bobina [uA]')
for a in ax:
    a.grid(alpha=.3)
ax[0].set_title(f"run {run_1t} | f CW inicial {d1['f_cw'][0]/1e9:.6f} GHz | {d1['metadata'].get('resonador', '')}")
fig.tight_layout()
plt.show()

### Cerrar

In [ ]:
sg.rf_output(False)
# fuente.rampa(0.0, paso=1e-6)      # descomentá para bajar la corriente de la bobina a 0
pna.close()
sg.close()
fuente.inst.close()